# Многопризнаковая регрессия на данных Diabetes

Обучим полносвязную сеть предсказывать числовое значение из набора Diabetes по десяти признакам. В отличие от игрушечного примера `y = 2x + 1`, здесь нет одной простой зависимости: сеть учится находить закономерности по нескольким входам.

Скрытый слой использует ReLU и Dropout. Выходной слой линейный, поэтому он может предсказывать любое вещественное число. Качество оценим на отложенных данных и сравним с простым прогнозом среднего значения целевой переменной обучающей выборки.


In [47]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from open_neuron.activations import linear, relu
from open_neuron.layers import Dropout, Layer
from open_neuron.models import NeuralNetwork
from open_neuron.optimizers import Adam

np.set_printoptions(precision=3, suppress=True)
diabetes = load_diabetes()
X = diabetes.data
y = diabetes.target.reshape(-1, 1)

print("Признаки:", len(diabetes.feature_names))
print("Пример имён признаков:", diabetes.feature_names[:3])
print("Размеры X и y:", X.shape, y.shape)
print("Диапазон целевых значений:", (y.min(), y.max()))


Признаки: 10
Пример имён признаков: ['age', 'sex', 'bmi']
Размеры X и y: (442, 10) (442, 1)
Диапазон целевых значений: (np.float64(25.0), np.float64(346.0))


## Разделяем и масштабируем данные

Сначала отделяем тестовую часть и задаём `random_state`, чтобы получить то же разбиение при повторном запуске. Затем `StandardScaler` находит средние и стандартные отклонения только по обучающим примерам. К тестовым данным применяется уже обученное преобразование, поэтому их статистики не просачиваются в подготовку обучения.

Целевые значения не масштабируем: метрики и предсказания останутся в исходном масштабе Diabetes.


In [48]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Обучающих примеров:", len(X_train))
print("Тестовых примеров:  ", len(X_test))
print("Средние признаков train после масштабирования:", X_train_scaled.mean(axis=0))


Обучающих примеров: 353
Тестовых примеров:   89
Средние признаков train после масштабирования: [ 0.  0. -0.  0. -0. -0.  0.  0.  0. -0.]


## Создаём и обучаем сеть

Сеть состоит из 10 нейронов ReLU, Dropout с долей отключения `0.2` и одного линейного выходного нейрона. Dropout применяется при обучении; во время `predict` слой пропускает вход без случайного отключения.

Мы не передаём тестовые данные в `fit`. Поэтому строка обучения с пометкой `on Train Data` относится к обучающим данным, а тестовые метрики считаются ниже один раз после обучения. Seed фиксирует и начальные параметры, и случайное перемешивание пакетов.


In [49]:
np.random.seed(7)
model = NeuralNetwork(
    [
        Layer(10, activation=relu),
        Dropout(0.2),
        Layer(1, activation=linear),
    ],
    inputs_dim=X_train_scaled.shape[1],
    loss="mse",
    optimizer=Adam(learning_rate=0.1),
)

model.fit(
    X_train_scaled,
    y_train,
    epochs=100,
    batch_size=8,
    final_batch_size=16,
    shuffle=True,
)


Training started with Overall Loss 30092.9829 on Train Data
Epoch 100/100, Loss: 3188.0717, Validation Loss: 2700.4881 on Train Data
Training completed with Overall Loss 2700.4881


## Оцениваем качество на тестовой выборке

MSE сильнее штрафует большие ошибки и измеряется в квадрате единиц цели. MAE показывает средний размер ошибки в исходных единицах. Для ориентира сравним сеть с базовым предсказанием: средним значением `y_train` для каждого тестового примера. Это помогает понять, выучила ли сеть что-то полезное относительно простого постоянного ответа.


In [53]:
predictions = model.predict(X_test_scaled)
baseline_predictions = np.full_like(y_test, y_train.mean())

mse = np.mean(np.square(predictions - y_test))
mae = np.mean(np.abs(predictions - y_test))
baseline_mse = np.mean(np.square(baseline_predictions - y_test))
baseline_mae = np.mean(np.abs(baseline_predictions - y_test))

print(f"Тестовая MSE сети:       {mse:.2f}")
print(f"Тестовая MAE сети:       {mae:.2f}")
print(f"Тестовая MSE среднего:   {baseline_mse:.2f}")
print(f"Тестовая MAE среднего:   {baseline_mae:.2f}")
print()
print("Первые предсказания и ответы:")
for expected, predicted in zip(y_test.ravel()[:5], predictions.ravel()[:5]):
    print(f"ответ: {expected:.1f}, предсказание: {predicted:.1f}")


Тестовая MSE сети:       2726.05
Тестовая MAE сети:       41.46
Тестовая MSE среднего:   5361.53
Тестовая MAE среднего:   64.01

Первые предсказания и ответы:
ответ: 219.0, предсказание: 133.0
ответ: 70.0, предсказание: 174.9
ответ: 202.0, предсказание: 141.1
ответ: 230.0, предсказание: 281.5
ответ: 111.0, предсказание: 106.8


В этом запуске меньшие MSE и MAE у сети, чем у базового среднего, означают, что модель использует признаки для более точных предсказаний. Числа зависят от выбранного разбиения и случайной инициализации; фиксированные seed позволяют воспроизвести именно этот учебный запуск.


In [51]:
model.summary()

Info:
Network type: "Regression", layers: 2, neurons: 11, optimizer: "Adam", epochs: 100, batch size: 8-16, loss function: "Mean Squared Error"
Last forward:
Layer 1, neurons: 10, inputs: [-1.407 -0.942  0.722 -0.497 -0.627 -0.379 -0.119 -0.062 -0.569 -1.176], activation: relu, output (a): [10.866  0.    17.246 12.497  9.653  9.509  0.     9.695  0.     1.369]
Unit 1/10, inputs: [-1.407 -0.942  0.722 -0.497 -0.627 -0.379 -0.119 -0.062 -0.569 -1.176], weights: [-2.961 -1.401  3.166  1.143 -2.392  4.22  -0.57   2.525  1.961  0.156], bias: [5.153], z: 10.866139736977953, activation: relu, a: 10.866139736977953
Unit 2/10, inputs: [-1.407 -0.942  0.722 -0.497 -0.627 -0.379 -0.119 -0.062 -0.569 -1.176], weights: [ 0.189  0.174 -0.191 -0.045 -0.52   0.197 -0.051 -0.584  0.033  0.096], bias: [-1.746], z: -2.130347283444787, activation: relu, a: 0.0
...
Unit 9/10, inputs: [-1.407 -0.942  0.722 -0.497 -0.627 -0.379 -0.119 -0.062 -0.569 -1.176], weights: [ 3.952  1.539  0.834  1.492 -4.121  1.35 